# LangChain Cache Reference

Developer-facing statements defined in `langchain_core.caches`.

# `RETURN_VAL_TYPE`

Type alias for values stored in and returned from language-model caches.

```python
RETURN_VAL_TYPE = Sequence[Generation]
```

A cached value is a sequence of `Generation` objects or subclasses.

---

# `BaseCache: ABC`

Abstract interface for caching language-model and chat-model generations.

Cache keys are derived from the combination of a serialized prompt and a serialized language-model configuration string.

## Required subclass hooks

### `lookup`

Looks up cached generations for a prompt and language-model configuration.

```python
@abstractmethod
lookup(
    self,
    prompt: str, # Serialized prompt or chat-model input
    llm_string: str, # Serialized language-model invocation configuration
) -> RETURN_VAL_TYPE | None # Cached generations, or None on a cache miss
```

A concrete subclass must implement this method. The abstract method body does not explicitly raise `NotImplementedError`.

### `update`

Stores generations for a prompt and language-model configuration.

```python
@abstractmethod
update(
    self,
    prompt: str, # Serialized prompt or chat-model input
    llm_string: str, # Serialized language-model invocation configuration
    return_val: RETURN_VAL_TYPE, # Generations to cache
) -> None
```

A concrete subclass must implement this method. The key-generation logic must be compatible with `lookup()`. The abstract method body does not explicitly raise `NotImplementedError`.

### `clear`

Clears cached values.

```python
@abstractmethod
clear(
    self,
    **kwargs: Any, # Cache-specific clearing options
) -> None
```

A concrete subclass must implement this method. The abstract method body does not explicitly raise `NotImplementedError`.

## Async wrappers

### `alookup`

Runs `lookup()` asynchronously through `run_in_executor()`.

```python
async alookup(
    self,
    prompt: str, # Serialized prompt or chat-model input
    llm_string: str, # Serialized language-model invocation configuration
) -> RETURN_VAL_TYPE | None # Cached generations, or None on a cache miss
```

The default implementation calls:

```python
await run_in_executor(None, self.lookup, prompt, llm_string)
```

Subclasses can override this method with a native asynchronous implementation.

### `aupdate`

Runs `update()` asynchronously through `run_in_executor()`.

```python
async aupdate(
    self,
    prompt: str, # Serialized prompt or chat-model input
    llm_string: str, # Serialized language-model invocation configuration
    return_val: RETURN_VAL_TYPE, # Generations to cache
) -> None
```

The default implementation calls:

```python
await run_in_executor(None, self.update, prompt, llm_string, return_val)
```

Subclasses can override this method with a native asynchronous implementation.

### `aclear`

Runs `clear()` asynchronously through `run_in_executor()`.

```python
async aclear(
    self,
    **kwargs: Any, # Cache-specific clearing options
) -> None
```

The default implementation forwards the keyword arguments to `clear()` through the executor. Subclasses can override this method with a native asynchronous implementation.

---



In [ ]:
from typing import Any # Import Any for cache-specific clear options

from langchain_core.caches import BaseCache, RETURN_VAL_TYPE # Import the abstract cache and cache value type
from langchain_core.outputs import Generation # Import Generation for cached model outputs


class DictionaryCache(BaseCache): # Create a concrete cache implementation
    def __init__(self) -> None: # Initialize the cache
        self.store: dict[tuple[str, str], RETURN_VAL_TYPE] = {} # Store values by prompt and configuration

    def lookup( # Implement synchronous cache lookup
        self,
        prompt: str, # Receive the serialized prompt
        llm_string: str, # Receive the serialized model configuration
    ) -> RETURN_VAL_TYPE | None:
        key = (prompt, llm_string) # Build the cache key
        return self.store.get(key) # Return cached generations or None

    def update( # Implement synchronous cache storage
        self,
        prompt: str, # Receive the serialized prompt
        llm_string: str, # Receive the serialized model configuration
        return_val: RETURN_VAL_TYPE, # Receive generations to cache
    ) -> None:
        key = (prompt, llm_string) # Build the cache key
        self.store[key] = return_val # Store generations under the key

    def clear(self, **kwargs: Any) -> None: # Implement cache clearing
        self.store.clear() # Remove every cached entry


cache = DictionaryCache() # Create the concrete cache

prompt = "Explain Python in one sentence." # Define the serialized prompt
llm_string = "model=gpt-example|temperature=0" # Define the model configuration

generations = [ # Create model generations to cache
    Generation(text="Python is a readable, general-purpose programming language."), # Create one response
]

initial_result = cache.lookup(prompt, llm_string) # Look up before storing
print("Initial lookup:", initial_result) # Display the cache miss

cache.update(prompt, llm_string, generations) # Store generations synchronously

cached_result = cache.lookup(prompt, llm_string) # Retrieve the stored generations
print("Cached text:", cached_result[0].text if cached_result else None) # Display cached text

different_config_result = cache.lookup( # Use the same prompt with another configuration
    prompt, # Reuse the prompt
    "model=gpt-example|temperature=1", # Use a different configuration
)

print("Different configuration:", different_config_result) # Display another cache miss

async_prompt = "What is LangChain?" # Define another serialized prompt
async_llm_string = "model=gpt-example|temperature=0.2" # Define another configuration

async_generations = [ # Create another cached response
    Generation(text="LangChain is a framework for building LLM applications."),
]

await cache.aupdate( # Use BaseCache's executor-backed async update
    async_prompt, # Provide the prompt
    async_llm_string, # Provide the model configuration
    async_generations, # Provide generations to cache
)

async_result = await cache.alookup(async_prompt, async_llm_string) # Look up asynchronously
print("Async cached text:", async_result[0].text if async_result else None) # Display the result

await cache.aclear() # Clear through the executor-backed async wrapper

result_after_clear = cache.lookup(prompt, llm_string) # Look up after clearing
print("After clear:", result_after_clear) # Display the cache miss

# `InMemoryCache: BaseCache`

Stores cached language-model generations in the current Python process.

Entries are keyed by the tuple `(prompt, llm_string)`. The cache can be unbounded or constrained by a maximum number of entries.

## Constructor

```python
InMemoryCache(
    *,
    maxsize: int | None = None, # Maximum entries, or None for no limit
) -> None
```

Raises `ValueError` with `"maxsize must be greater than 0"` when `maxsize` is not `None` and is less than or equal to zero.

## Methods

### `lookup`

Returns the value stored for a prompt and language-model configuration.

```python
lookup(
    self,
    prompt: str, # Serialized prompt or chat-model input
    llm_string: str, # Serialized language-model invocation configuration
) -> RETURN_VAL_TYPE | None # Cached generations, or None on a cache miss
```

The method performs a dictionary lookup using `(prompt, llm_string)`.

### `update`

Stores a cache value.

```python
update(
    self,
    prompt: str, # Serialized prompt or chat-model input
    llm_string: str, # Serialized language-model invocation configuration
    return_val: RETURN_VAL_TYPE, # Generations to cache
) -> None
```

When `maxsize` is set and the cache currently contains exactly that number of entries, the first inserted dictionary entry is removed before the new value is stored.

The supplied sequence is stored directly without copying.

### `clear`

Removes every cached entry.

```python
@override
clear(
    self,
    **kwargs: Any, # Accepted but ignored
) -> None
```

The internal cache dictionary is replaced with a new empty dictionary.

### `alookup`

Asynchronously returns a cached value.

```python
async alookup(
    self,
    prompt: str, # Serialized prompt or chat-model input
    llm_string: str, # Serialized language-model invocation configuration
) -> RETURN_VAL_TYPE | None # Cached generations, or None on a cache miss
```

This override directly calls the synchronous `lookup()` method without using an executor.

### `aupdate`

Asynchronously stores a cache value.

```python
async aupdate(
    self,
    prompt: str, # Serialized prompt or chat-model input
    llm_string: str, # Serialized language-model invocation configuration
    return_val: RETURN_VAL_TYPE, # Generations to cache
) -> None
```

In [ ]:
from langchain_core.caches import InMemoryCache # Import the real in-memory cache
from langchain_core.outputs import Generation # Import Generation for cached model outputs


cache = InMemoryCache(maxsize=2) # Create a cache that can hold at most two entries

prompt_one = "What is Python?" # Define the first serialized prompt
config_one = "model=gpt-example|temperature=0" # Define the first model configuration
result_one = [Generation(text="Python is a general-purpose programming language.")] # Create the first cached result

cache.update(prompt_one, config_one, result_one) # Store the first cache entry

cached_one = cache.lookup(prompt_one, config_one) # Retrieve the first cache entry
print("First cached result:", cached_one[0].text if cached_one else None) # Display the cached text

missing_result = cache.lookup("Unknown prompt", config_one) # Look up a key that is not stored
print("Missing result:", missing_result) # Display None for the cache miss

prompt_two = "What is LangChain?" # Define the second serialized prompt
config_two = "model=gpt-example|temperature=0.2" # Define the second model configuration
result_two = [Generation(text="LangChain is a framework for building LLM applications.")] # Create the second cached result

cache.update(prompt_two, config_two, result_two) # Store the second cache entry

prompt_three = "What is caching?" # Define the third serialized prompt
config_three = "model=gpt-example|temperature=0.5" # Define the third model configuration
result_three = [Generation(text="Caching stores reusable results for faster access.")] # Create the third cached result

cache.update(prompt_three, config_three, result_three) # Store the third entry and evict the first entry

evicted_result = cache.lookup(prompt_one, config_one) # Check whether the first entry was evicted
print("First result after eviction:", evicted_result) # Display None after eviction

second_result = cache.lookup(prompt_two, config_two) # Retrieve the second entry
print("Second cached result:", second_result[0].text if second_result else None) # Display the second cached text

async_prompt = "What is an async cache method?" # Define a prompt for asynchronous operations
async_config = "model=gpt-example|temperature=0.1" # Define its model configuration
async_value = [Generation(text="It is an awaitable cache operation.")] # Create the asynchronous cache value

await cache.aupdate(async_prompt, async_config, async_value) # Store the value asynchronously

async_result = await cache.alookup(async_prompt, async_config) # Retrieve the value asynchronously
print("Async cached result:", async_result[0].text if async_result else None) # Display the asynchronous result

await cache.aclear() # Clear every cache entry asynchronously

cleared_result = cache.lookup(async_prompt, async_config) # Look up an entry after clearing
print("Result after clear:", cleared_result) # Display None after clearing

try: # Start invalid maxsize handling
    InMemoryCache(maxsize=0) # Try creating a cache with an invalid maximum size
except ValueError as error: # Catch the expected validation error
    print("Invalid maxsize error:", error) # Display the error message